# 14 — Full DeepSeek-V2 tiny model (phase 3)

**Before:** notebooks **12–13** (MLA + MoE).

**After:** **15** train, **16** sample.

Same role as notebook **6.GPT.ipynb**, but each block is:

```text
x = x + MLA(RMSNorm(x))
x = x + MoE(RMSNorm(x))
```

**C port:** `c/deepseek_v2/block.c` — `make test_block`.


In [ ]:
import torch
from llmc.data import CharTokenizer, load_text
from llmc.deepseek_v2 import DeepSeekV2, DeepSeekV2Config

text = load_text("data/tiny_shakespeare.txt")
tok = CharTokenizer.from_text(text)
cfg = DeepSeekV2Config.tiny(tok.vocab_size, block_size=64)
model = DeepSeekV2(cfg)

print(model)
print("parameters:", f"{model.count_parameters():,}")


In [ ]:
# One forward pass (like notebook 6)
x = torch.randint(0, tok.vocab_size, (4, 32))
logits, loss = model(x, x)
print("logits", tuple(logits.shape), "| loss", round(loss.item(), 4))


In [ ]:
# Inspect block 0 — names you will grep in C
b0 = model.blocks[0]
for name, mod in b0.named_children():
    print(name, "->", mod.__class__.__name__)


In [ ]:
# Trace shapes through block 0 manually (hand-holding)
idx = torch.randint(0, tok.vocab_size, (2, 16))
h = model.tok_emb(idx)
print("embed:", tuple(h.shape))

x = h
x = x + b0.attn(b0.ln1(x))
print("after MLA residual:", tuple(x.shape))
x = x + b0.moe(b0.ln2(x))
print("after MoE residual:", tuple(x.shape))


## C exercise

```bash
cd c && make test_block && ./bin/test_block
```

Read `block.c` next to `DeepSeekV2Block.forward` in Python.

**Next:** notebook **15** — training loop (reuse `llmc.train.Trainer`).


In [ ]:
import shutil, subprocess
if shutil.which("make"):
    r = subprocess.run(["make", "test_block"], cwd="c", capture_output=True, text=True)
    print(r.stdout or r.stderr)
